# SSPNet — PTB-XL NORM / IMI / ASMI: multi-seed training, EMA weights, seed ensemble, validation-tuned decision rule (CPU)

**Inputs and architecture are the same as V4** (signed median-beat traces + 8–40 Hz spectrograms, 16×16 attention). V5 changes *how the model is trained and how its outputs are used*, because the V3 runs showed two problems:

1. **Large seed-to-seed variation.** Different seeds gave noticeably different validation results. Picking the luckiest seed is cherry-picking and will not reproduce. V5 instead trains several seeds (`SEEDS`) and **averages their probabilities (seed ensemble)**. This is a real, reportable improvement: ensembles reduce variance and usually raise AUC.
2. **Noisy validation curves and overfitting** (training loss keeps falling, validation loss flat from epoch ~9, best epoch decided by noise on only 34 ASMI records). V5 keeps an **exponential moving average (EMA) of the weights** and evaluates/saves the EMA model, which smooths the curves and generalises better. Gradient clipping is added for stability.
3. **Argmax under class imbalance under-predicts MI** (V3: IMI recall 0.40, ASMI 0.47). V5 learns **one additive log-probability offset per class on validation fold 9 only**, maximising macro-F1, and applies that fixed rule to the test fold. It changes the decision threshold, not the model.

Fold 10 is still evaluated once, at the end, with everything frozen. Each seed has its own run directory (`runs/mi_sspnet_v5_s<seed>/`), the ensemble writes to `runs/mi_sspnet_v5_ensemble/`. V3/V4 files are not touched; the V4 image cache is reused.

**Honest status:** the PyTorch cells could not be executed in the authoring environment (no torch). The numpy-only parts (ensemble/offset tuning) were tested on synthetic data. Fix any error you hit and report it.

## 0. Setup and instructions

1. Set `PTBXL_ROOT`. The V4 image cache (`ptbxl_cache_v4_signedbeat/`, ~3.9 GiB) is built on the first run and reused afterwards.
2. **Restart the kernel and Run All.** With `RUN_TRAINING = True`, the training cell trains every seed in `SEEDS` one after another. Each seed resumes automatically if interrupted, so you can stop the notebook and rerun it later (e.g. overnight).
3. Section 6b builds the **seed ensemble** from all finished seeds and tunes the class offsets on fold 9. It only reads saved validation predictions, so it is fast and can be rerun as more seeds finish.
4. When all decisions are frozen, set `EVALUATE_TEST = True` and run Section 7 **once**. It evaluates every seed and the ensemble on fold 10.
5. Sections 8–9 produce the figures and statistics for the ensemble (and compare it with V3).

**Do not change hyperparameters inside existing run directories — change `RUN_PREFIX` instead.** Do not choose seeds by their test-fold score.

In [ ]:
# If necessary, run this once in a Jupyter cell (uncomment):
# %pip install -U numpy scipy pandas matplotlib scikit-learn wfdb torch ipython

In [ ]:
from pathlib import Path
import os, sys, json, time, random, hashlib, shutil, copy
import numpy as np
import pandas as pd
import torch

# ---- EDIT ROOT FOLDER (existing PTB-XL download, NO redownload) ----
PTBXL_ROOT = Path(os.environ.get("PTBXL_ROOT", "path/to/ptb-xl/1.0.3"))
CACHE_DIR = Path('ptbxl_cache_v4_signedbeat')   # V5 reuses the V4 cache; never point this at the V1/V3 caches
RUN_PREFIX = 'runs/mi_sspnet_v5'                 # seed runs -> runs/mi_sspnet_v5_s1, _s2, ...; ensemble -> _ensemble
ENSEMBLE_DIR = Path(RUN_PREFIX + '_ensemble')
V3_BASELINE_RUN = Path('runs/mi_sspnet_12lead_v3_s1')   # for the paired comparison in Section 9

MODEL_KIND = 'sspnet'     # 'sspnet' or 'cnn'
SEEDS = [1, 2, 3]         # 3 is the minimum for an ensemble + mean +- SD; 5 is better if you have time
RUN_TRAINING = True
RESUME = True
EVALUATE_TEST = False     # set True ONLY after all model-selection decisions are frozen

# CPU-safe defaults.
CPU_THREADS = 4
NUM_WORKERS = 0
BATCH_SIZE = 8
EPOCHS = 40
LR = 1e-4
WEIGHT_DECAY = 1e-3
DROPOUT = 0.4
LABEL_SMOOTHING = 0.05
CLASS_WEIGHTING = 'sqrt'   # 'sqrt', 'inverse', 'none'
MONITOR = 'val_macro_auc'  # checkpoint criterion ('val_loss', 'val_macro_f1' also supported)
TOKEN_POOL = 2             # 1 = 32x32 attention grid (V3), 2 = 16x16
AUG_SHIFT = 6              # train only: random time shift (+/- pixels)
AUG_TIMEMASK = 12          # train only: random time mask on the spectrogram branch (max width, pixels)
PATIENCE = 10
SCHEDULER_PATIENCE = 2
SCHEDULER_FACTOR = 0.5
# ---- V5 additions ----
USE_EMA = True             # evaluate and save an exponential moving average of the weights
EMA_DECAY = 0.999          # ~1000-step horizon = about one epoch at batch size 8
GRAD_CLIP = 1.0            # max gradient norm (0 = off)
BIAS_GRID = np.round(np.arange(-1.0, 3.01, 0.1), 2)   # search range for the per-class log-offsets (Section 6b)

PROGRESS_EVERY = 250

DEVICE = torch.device('cpu')
torch.set_num_threads(CPU_THREADS)
try:
    torch.set_num_interop_threads(min(4, CPU_THREADS))
except RuntimeError:
    pass

CLASS_NAMES = ['NORM', 'IMI', 'ASMI']
EXPECTED_IMAGE_SHAPE = (2, 12, 128, 128)
CACHE_SCHEMA = 'sspnet-v4-signedbeat-v1'

def run_dir_for(seed):
    return Path(f'{RUN_PREFIX}_s{seed}')

def set_all_seeds(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

SEED = SEEDS[0]; RUN_DIR = run_dir_for(SEED); set_all_seeds(SEED)

print('PyTorch:', torch.__version__, '| device:', DEVICE, '| CPU threads:', torch.get_num_threads())
print('PTB-XL:', PTBXL_ROOT, '| cache:', CACHE_DIR)
print('seed runs:', [str(run_dir_for(s)) for s in SEEDS], '| ensemble:', ENSEMBLE_DIR)
print('Free disk: %.1f GiB' % (shutil.disk_usage(Path.cwd()).free / 1024**3))
assert (PTBXL_ROOT / 'ptbxl_database.csv').is_file() or (CACHE_DIR / 'cache_config.json').is_file(), (
    'Set PTBXL_ROOT to the folder with ptbxl_database.csv, unless a complete V4 cache already exists.')

## 1. SSPNet architecture — V3 model plus one option: `token_pool`

Convolution blocks, channel/spatial attention, bidirectional cross-attention, sum/difference fusion, bottleneck and multi-head self-attention are as in V3. The only change: after each branch the 256×32×32 feature map can be average-pooled (`token_pool=2` → 256×16×16), so attention runs on **256 tokens instead of 1024**. `token_pool=1` reproduces V3 exactly. The optional BaselineCNN is unchanged (24 channels).

In [ ]:
"""
SSPNet (Fig. 2): dual-branch spatio-spectral portrait network.

  alpha (12 signed median-beat traces, 128x128) -> vertical   ACB branch  -> 256x32x32 ┐
                                                           ├ cross-attention -> sum / diff -> concat(512)
  delta (12-lead 8-40 Hz spectrograms)    -> horizontal ACB branch  -> 256x32x32 ┘   -> conv(256) -> bottleneck
                                                               -> multi-head attention -> FC(512) -> {NORM, IMI, ASMI}

Feature-map sizes per branch follow the figure:
  64x64x64 -> 128x64x64 (+channel attn) -> 128x32x32 -> 256x32x32 (+spatial attn)
"""
import torch
import torch.nn as nn


class ACB(nn.Module):
    """Asymmetric convolution block: square kxk conv + (kx1 | 1xk) conv, summed, then BN + ReLU.
    orient='v' -> vertical kernel (K x 1) for the alpha band (upper/lower asymmetry)
    orient='h' -> horizontal kernel (1 x K) for the delta band (left/right asymmetry)"""

    def __init__(self, cin, cout, orient, stride=1, k=3):
        super().__init__()
        self.square = nn.Conv2d(cin, cout, k, stride, padding=k // 2, bias=False)
        if orient == "v":
            self.asym = nn.Conv2d(cin, cout, (k, 1), stride, padding=(k // 2, 0), bias=False)
        elif orient == "h":
            self.asym = nn.Conv2d(cin, cout, (1, k), stride, padding=(0, k // 2), bias=False)
        else:
            raise ValueError(orient)
        self.bn = nn.BatchNorm2d(cout)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.act(self.bn(self.square(x) + self.asym(x)))


class ChannelAttention(nn.Module):
    def __init__(self, c, r=8):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(c, c // r), nn.ReLU(inplace=True), nn.Linear(c // r, c))

    def forward(self, x):
        a = self.mlp(x.mean((2, 3))) + self.mlp(x.amax((2, 3)))
        return x * torch.sigmoid(a)[:, :, None, None]


class SpatialAttention(nn.Module):
    def __init__(self, k=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, k, padding=k // 2)

    def forward(self, x):
        s = torch.cat([x.mean(1, keepdim=True), x.amax(1, keepdim=True)], 1)
        return x * torch.sigmoid(self.conv(s))


class Branch(nn.Module):
    def __init__(self, orient, in_ch=12):
        super().__init__()
        self.b1 = ACB(in_ch, 64, orient, stride=2)    # 64 x 64 x 64
        self.b2 = ACB(64, 128, orient)                # 128 x 64 x 64
        self.ca = ChannelAttention(128)
        self.b3 = ACB(128, 128, orient, stride=2)     # 128 x 32 x 32
        self.b4 = ACB(128, 256, orient)               # 256 x 32 x 32
        self.sa = SpatialAttention()

    def forward(self, x):
        x = self.ca(self.b2(self.b1(x)))
        return self.sa(self.b4(self.b3(x)))


class CrossAttention(nn.Module):
    """Tokens = spatial positions of one branch (query) attending to the other branch (key/value)."""

    def __init__(self, dim=256, heads=4, n_tokens=32 * 32):
        super().__init__()
        self.pos = nn.Parameter(torch.zeros(1, n_tokens, dim))
        nn.init.trunc_normal_(self.pos, std=0.02)
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm = nn.LayerNorm(dim)

    def forward(self, q, kv):
        q, kv = q + self.pos, kv + self.pos
        out, _ = self.attn(q, kv, kv, need_weights=False)
        return self.norm(q + out)


class Bottleneck(nn.Module):
    def __init__(self, c=256, mid=64):
        super().__init__()
        self.f = nn.Sequential(
            nn.Conv2d(c, mid, 1, bias=False), nn.BatchNorm2d(mid), nn.ReLU(inplace=True),
            nn.Conv2d(mid, mid, 3, padding=1, bias=False), nn.BatchNorm2d(mid), nn.ReLU(inplace=True),
            nn.Conv2d(mid, c, 1, bias=False), nn.BatchNorm2d(c),
        )
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.act(x + self.f(x))


class SSPNet(nn.Module):
    def __init__(self, num_classes=3, dim=256, heads=4, mha_heads=8, dropout=0.3, token_pool=1):
        super().__init__()
        assert 32 % token_pool == 0
        self.pool = nn.AvgPool2d(token_pool) if token_pool > 1 else nn.Identity()
        n_tok = (32 // token_pool) ** 2
        self.alpha = Branch("v", in_ch=12)   # alpha band: vertical asymmetric kernel
        self.delta = Branch("h", in_ch=12)   # delta band: horizontal asymmetric kernel
        self.xa_alpha = CrossAttention(dim, heads, n_tokens=n_tok)   # alpha queries, delta keys/values
        self.xa_delta = CrossAttention(dim, heads, n_tokens=n_tok)   # delta queries, alpha keys/values
        self.fuse = nn.Sequential(nn.Conv2d(2 * dim, dim, 1, bias=False), nn.BatchNorm2d(dim), nn.ReLU(inplace=True))
        self.bottleneck = Bottleneck(dim)
        self.mha_norm = nn.LayerNorm(dim)
        self.mha = nn.MultiheadAttention(dim, mha_heads, batch_first=True)
        self.head = nn.Sequential(
            nn.Linear(dim, 512), nn.ReLU(inplace=True), nn.Dropout(dropout), nn.Linear(512, num_classes)
        )

    @staticmethod
    def _tok(x):    # (B,C,H,W) -> (B,HW,C)
        return x.flatten(2).transpose(1, 2)

    def forward(self, alpha, delta):
        fa, fd = self.pool(self.alpha(alpha)), self.pool(self.delta(delta))   # (B,256,32/p,32/p) each
        B, C, H, W = fa.shape
        ta, td = self._tok(fa), self._tok(fd)
        ra = self.xa_alpha(ta, td)                               # alpha refined by delta
        rd = self.xa_delta(td, ta)                               # delta refined by alpha
        shared = ra + rd                                         # element-wise summation (shared)
        contrast = rd - ra                                       # element-wise subtraction (delta - alpha)
        x = torch.cat([shared, contrast], dim=-1)                # (B,HW,512)
        x = x.transpose(1, 2).reshape(B, 2 * C, H, W)
        x = self.bottleneck(self.fuse(x))                        # 512ch -> 256ch -> bottleneck
        t = self._tok(x)
        n = self.mha_norm(t)
        t = t + self.mha(n, n, n, need_weights=False)[0]         # multi-head self-attention
        return self.head(t.mean(1))


class BaselineCNN(nn.Module):
    """Plain CNN baseline / ablation on the same images: the two band images are stacked into 24 channels,
    with no asymmetric kernels, no attention and no cross-attention."""

    def __init__(self, num_classes=3, dropout=0.3):
        super().__init__()

        def blk(i, o):
            return nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o),
                                 nn.ReLU(inplace=True), nn.MaxPool2d(2))

        self.features = nn.Sequential(blk(24, 64), blk(64, 128), blk(128, 256), blk(256, 256))
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(256, num_classes))

    def forward(self, alpha, delta):
        x = self.features(torch.cat([alpha, delta], 1))
        return self.head(x.mean((2, 3)))

In [ ]:
# CPU smoke test: this is NOT training and does not access the PTB-XL dataset.
# Set to False if you already verified the forward pass on this machine.
RUN_SMOKE_TEST = True
if RUN_SMOKE_TEST:
    for _tp in (1, TOKEN_POOL):
        trial = SSPNet(num_classes=3, dropout=DROPOUT, token_pool=_tp).to('cpu').eval()
        with torch.inference_mode():
            out = trial(torch.zeros(1, 12, 128, 128), torch.zeros(1, 12, 128, 128))
        assert tuple(out.shape) == (1, 3)
        print(f'SSPNet forward pass OK (token_pool={_tp}):', tuple(out.shape),
              f'{sum(p.numel() for p in trial.parameters())/1e6:.2f}M params')
    del trial, out

## 2. Preprocessing — signed median-beat traces + 12-lead high-band spectrograms

Per record (`records100`, 100 Hz, 10 s), output `(2, 12, 128, 128)` uint8:

- **channel 0 – median-beat trace (alpha branch):** Butterworth band-pass 0.5–40 Hz (zero-phase) → R-peaks from the lead-combined 5–18 Hz QRS envelope → median of all complete beats (0.3 s before to 0.6 s after the R peak) per lead → PR-segment baseline subtraction → drawn as a 128×128 image: columns = time, rows = voltage on a **fixed ±1.5 mV scale** (up = positive). White line, grey fill between the line and 0 mV, so ST elevation vs depression and Q/T polarity are directly visible. If no beat is found, the strongest-QRS-energy window is used (`n_beats = 0` in `meta.csv`).
- **channel 1 – high band (delta branch):** identical to V3 (STFT, `log1p`, 8–40 Hz, per-lead min–max).

**Cohort:** identical to V3 (same function, same row order). The cache build checks this against `ptbxl_cache_12lead_v3/meta.csv` if that folder exists. Limitations: the median beat assumes a regular rhythm; arrhythmias and pacing reduce its quality (see `n_beats` in `meta.csv`); the ±1.5 mV scale clips very large QRS peaks.

In [ ]:
import ast
from scipy.ndimage import zoom
from scipy.signal import stft

FS = 100
NPERSEG, NOVERLAP, NFFT = 128, 120, 512
BANDS = [(0.5, 8.0), (8.0, 40.0)]
IMG = 128
LEADS = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
DIAGNOSTIC_CLASSES = ['NORM', 'MI', 'STTC', 'CD', 'HYP']


def build_labels_original_rules(root):
    """Same `diagnostic==1`, single-superclass, and fine-SCP-code rules as V1/V2."""
    db = pd.read_csv(Path(root) / 'ptbxl_database.csv', index_col='ecg_id')
    db['scp_codes'] = db['scp_codes'].apply(ast.literal_eval)
    agg = pd.read_csv(Path(root) / 'scp_statements.csv', index_col=0)
    diag = agg[agg['diagnostic'] == 1]['diagnostic_class'].to_dict()
    db['codes'] = db['scp_codes'].apply(lambda c: '|'.join(sorted(k for k in c if k in diag)))
    db['supers'] = db['scp_codes'].apply(lambda c: sorted({diag[k] for k in c if k in diag}))
    db = db[db['supers'].apply(len) == 1].copy()
    db['class'] = db['supers'].str[0]
    db['label'] = db['class'].map({c: i for i, c in enumerate(DIAGNOSTIC_CLASSES)})
    return db


def select_v3_cohort(root):
    db = build_labels_original_rules(root)
    targets = set(CLASS_NAMES)
    keep = db['codes'].apply(lambda s: len(s.split('|')) == 1 and s in targets)
    db = db[keep].copy()
    db['label'] = db['codes'].map({name: k for k, name in enumerate(CLASS_NAMES)}).astype(int)
    # Include IDs to check test record alignment across models.
    df = db.reset_index()[['ecg_id','filename_lr','label','class','strat_fold','codes']]
    if df.empty or not set(df['label']) == set(range(3)):
        raise ValueError('Cohort is empty or missing classes; inspect PTB-XL metadata.')
    if df['ecg_id'].duplicated().any():
        raise ValueError('Duplicate ecg_id in selected cohort.')
    return df


def ecg_to_12lead_portraits(sig, fs=FS):
    """(1000,12) mV ECG -> uint8 (2,12,128,128), 12 leads kept separate."""
    sig = np.asarray(sig)
    if sig.ndim != 2 or sig.shape[1] != 12 or sig.shape[0] < NPERSEG:
        raise ValueError(f'Expected ECG array (samples >= {NPERSEG}, 12), got {sig.shape}')
    f, _, Z = stft(sig.T, fs=fs, nperseg=NPERSEG, noverlap=NOVERLAP, nfft=NFFT)
    mag = np.log1p(np.abs(Z))  # (12, frequencies, frames)
    out = np.empty((2, 12, IMG, IMG), dtype=np.uint8)
    for b, (lo, hi) in enumerate(BANDS):
        rows = np.where((f >= lo) & (f < hi))[0]
        if len(rows) == 0:
            raise ValueError(f'No STFT bins for band {lo}-{hi} Hz')
        band = mag[:, rows, :][:, ::-1, :]  # same orientation as original
        # Treat the lead/channel axis as discrete; DO NOT interpolate across leads.
        resized = zoom(band, (1.0, IMG / band.shape[1], IMG / band.shape[2]), order=1)
        assert resized.shape == (12, IMG, IMG)
        lo_c = resized.min(axis=(1,2), keepdims=True)
        hi_c = resized.max(axis=(1,2), keepdims=True)
        out[b] = np.clip(255.0 * (resized - lo_c) / (hi_c - lo_c + 1e-12), 0, 255).astype(np.uint8)
    return out


def read_ecg(root, rel):
    import wfdb
    import numpy as np
    from pathlib import Path

    # Read the ECG recording
    sig, info = wfdb.rdsamp(str(Path(root) / str(rel)))

    # Get actual lead names
    names = info.get('sig_name', [])

    # Expected 12-lead ECG order
    expected_leads = [
        'I', 'II', 'III',
        'aVR', 'aVL', 'aVF',
        'V1', 'V2', 'V3',
        'V4', 'V5', 'V6'
    ]

    # Compare without case sensitivity
    actual_upper = [str(name).upper() for name in names]
    expected_upper = [name.upper() for name in expected_leads]

    if actual_upper != expected_upper:
        raise ValueError(
            f"Unexpected lead order: {names}. "
            f"Expected: {expected_leads}"
        )

    # Verify sampling frequency
    if int(round(info.get('fs', -1))) != FS:
        raise ValueError(
            f"Expected sampling rate {FS} Hz, "
            f"got {info.get('fs')} Hz for {rel}"
        )

    # Handle missing or infinite values
    sig = np.nan_to_num(
        sig,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    return sig


# ---- V4 additions: signed median-beat trace images --------------------------------------------
from scipy.signal import butter, sosfiltfilt, find_peaks

PRE, POST = 30, 60                 # samples before / after the R peak (0.30 s / 0.60 s at 100 Hz)
BEAT_LEN = PRE + POST
AMP_MV = 1.5                       # fixed vertical scale: image rows span -AMP_MV..+AMP_MV millivolts
_SOS_ECG = butter(3, [0.5, 40.0], btype='bandpass', fs=FS, output='sos')
_SOS_QRS = butter(2, [5.0, 18.0], btype='bandpass', fs=FS, output='sos')


def detect_r_peaks(sig_f, fs=FS):
    """sig_f: band-passed (n, 12). Returns (peak sample indices, lead-combined QRS envelope)."""
    qrs = sosfiltfilt(_SOS_QRS, sig_f, axis=0)
    env = np.sqrt((qrs ** 2).sum(1))
    k = max(1, int(0.08 * fs))
    env = np.convolve(env, np.ones(k) / k, mode='same')
    top = np.percentile(env, 99)
    if not np.isfinite(top) or top <= 1e-6:
        return np.array([], dtype=int), env
    peaks, _ = find_peaks(env, height=0.35 * top, distance=int(0.30 * fs))
    return peaks, env


def median_beat(sig_f, peaks, env):
    """Median over all complete beats, per lead, with the PR segment as the zero level -> (BEAT_LEN, 12), n_beats."""
    n = len(sig_f)
    ok = [p for p in peaks if p - PRE >= 0 and p + POST <= n]
    if ok:
        beats = np.stack([sig_f[p - PRE:p + POST] for p in ok])          # (nb, BEAT_LEN, 12)
        beat = np.median(beats, axis=0)
    else:                                                                  # fallback: strongest QRS-energy sample
        c = int(np.clip(np.argmax(env), PRE, n - POST))
        beat = sig_f[c - PRE:c + POST].copy()
    beat = beat - beat[PRE - 12:PRE - 4].mean(0, keepdims=True)           # PR-segment (~120-40 ms before R) baseline
    return beat, len(ok)


def beat_to_trace_images(beat):
    """(BEAT_LEN, 12) mV -> uint8 (12, IMG, IMG). Row 0 = +AMP_MV (up = positive), columns = time.
    255 = trace line, 96 = area between trace and 0 mV, 0 = background. Polarity and amplitude are preserved."""
    out = np.zeros((12, IMG, IMG), dtype=np.uint8)
    t_src, t_dst = np.arange(len(beat)), np.linspace(0, len(beat) - 1, IMG)
    zero = int(round((IMG - 1) / 2))
    for l in range(12):
        v = np.interp(t_dst, t_src, beat[:, l])
        r = np.clip(np.rint((AMP_MV - v) / (2 * AMP_MV) * (IMG - 1)), 0, IMG - 1).astype(int)
        for c in range(IMG):
            lo, hi = sorted((r[c], zero))
            out[l, lo:hi + 1, c] = 96                                      # signed area
        for c in range(IMG):
            a = r[c] if c == 0 else r[c - 1]
            lo, hi = sorted((a, r[c]))
            out[l, max(lo - 1, 0):min(hi + 1, IMG - 1) + 1, c] = 255        # line, joined to previous column
    return out


def ecg_to_v4_portraits(sig, fs=FS):
    """(1000, 12) mV ECG -> (uint8 (2, 12, 128, 128), n_beats).
    channel 0 (alpha branch): signed median-beat trace per lead; channel 1 (delta branch): 8-40 Hz STFT, as in V3."""
    sig = np.asarray(sig, dtype=np.float64)
    if sig.ndim != 2 or sig.shape[1] != 12 or sig.shape[0] < NPERSEG:
        raise ValueError(f'Expected ECG array (samples >= {NPERSEG}, 12), got {sig.shape}')
    sig_f = sosfiltfilt(_SOS_ECG, sig, axis=0)
    peaks, env = detect_r_peaks(sig_f, fs)
    beat, nb = median_beat(sig_f, peaks, env)
    out = np.empty((2, 12, IMG, IMG), dtype=np.uint8)
    out[0] = beat_to_trace_images(beat)
    out[1] = ecg_to_12lead_portraits(sig, fs)[1]
    return out, nb


def build_v4_cache():
    """Crash-safe cache: a partial file is NEVER treated as a completed cache."""
    if not (PTBXL_ROOT / 'ptbxl_database.csv').exists():
        raise FileNotFoundError('Cannot preprocess: set PTBXL_ROOT to the downloaded PTB-XL directory.')
    df = select_v3_cohort(PTBXL_ROOT)      # same cohort and order as V3
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    partial = CACHE_DIR / '.images_building.npy'
    final = CACHE_DIR / 'images.npy'
    for stale in [partial, CACHE_DIR / 'meta.csv', CACHE_DIR / 'cache_config.json']:
        if stale.exists():
            stale.unlink()
    n = len(df)
    print(f'Processing {n} recordings; estimated cache ~{n * np.prod(EXPECTED_IMAGE_SHAPE) / 1024**3:.2f} GiB')
    assert shutil.disk_usage(CACHE_DIR).free > n * np.prod(EXPECTED_IMAGE_SHAPE) * 1.08, (
        'Insufficient free disk space for V4 images. Move CACHE_DIR to a larger drive.')
    arr = np.lib.format.open_memmap(partial, mode='w+', dtype='uint8', shape=(n, *EXPECTED_IMAGE_SHAPE))
    n_beats = np.zeros(n, dtype=np.int64)
    t0 = time.time()
    try:
        for i, rel in enumerate(df['filename_lr']):
            arr[i], n_beats[i] = ecg_to_v4_portraits(read_ecg(PTBXL_ROOT, rel))
            if (i + 1) % PROGRESS_EVERY == 0 or i + 1 == n:
                print(f'V4 images: {i+1:,}/{n:,}  ({time.time()-t0:.0f}s)', flush=True)
        arr.flush()
    finally:
        del arr
    os.replace(partial, final)
    df = df.assign(n_beats=n_beats)
    df.to_csv(CACHE_DIR / 'meta.csv', index=False)
    manifest = dict(schema=CACHE_SCHEMA, expected_shape=list(EXPECTED_IMAGE_SHAPE),
                    n_records=n, dtype='uint8', fs=FS, lead_order=LEADS,
                    channel0='signed median-beat trace image (PRE=%d, POST=%d samples, +/-%.1f mV, 0.5-40 Hz band-pass)' % (PRE, POST, AMP_MV),
                    channel1='12-lead 8-40 Hz STFT magnitude (as V3)',
                    nperseg=NPERSEG, noverlap=NOVERLAP, nfft=NFFT,
                    cohort='single superclass + exactly one diagnostic code in NORM/IMI/ASMI (same as V3)',
                    data_source=str(PTBXL_ROOT.resolve()))
    (CACHE_DIR / 'cache_config.json').write_text(json.dumps(manifest, indent=2))
    print('V4 cache complete:', CACHE_DIR.resolve())


def verify_v4_cache():
    for name in ['images.npy', 'meta.csv', 'cache_config.json']:
        if not (CACHE_DIR / name).is_file():
            return False
    manifest = json.loads((CACHE_DIR / 'cache_config.json').read_text())
    if manifest.get('schema') != CACHE_SCHEMA or manifest.get('expected_shape') != list(EXPECTED_IMAGE_SHAPE):
        raise ValueError('Cache manifest is not a V4 signed-beat cache. Use a different CACHE_DIR.')
    im = np.load(CACHE_DIR / 'images.npy', mmap_mode='r')
    df = pd.read_csv(CACHE_DIR / 'meta.csv')
    if im.dtype != np.uint8 or tuple(im.shape) != (len(df), *EXPECTED_IMAGE_SHAPE):
        raise ValueError(f'Corrupt/incomplete cache: images={im.shape}/{im.dtype}, metadata={len(df)}')
    if manifest['n_records'] != len(df):
        raise ValueError('Cache manifest and metadata record counts disagree.')
    return True

In [ ]:
if verify_v4_cache():
    print('Verified completed V4 cache; skipping preprocessing:', CACHE_DIR)
else:
    if (CACHE_DIR / 'images.npy').exists():
        raise RuntimeError('images.npy exists without valid V4 manifest. Move/remove that incomplete cache manually.')
    build_v4_cache()
    assert verify_v4_cache()

# Same cohort as V3? (needed for paired V4-vs-V3 statistics)
_v3meta = Path('ptbxl_cache_12lead_v3/meta.csv')
if _v3meta.is_file():
    _a, _b = pd.read_csv(_v3meta), pd.read_csv(CACHE_DIR / 'meta.csv')
    assert _a[['ecg_id', 'label', 'strat_fold']].equals(_b[['ecg_id', 'label', 'strat_fold']]), 'V4 cohort differs from V3!'
    print('Cohort identical to V3 (ecg_id, label, fold): paired comparison is valid.')
else:
    print('V3 cache not found; cohort identity with V3 not checked.')

In [ ]:
# Visual quality check of the signed beat images (channel 0): one example per class, 4 leads each.
import matplotlib.pyplot as plt
_im = np.load(CACHE_DIR / 'images.npy', mmap_mode='r'); _m = pd.read_csv(CACHE_DIR / 'meta.csv')
print('beats found per record: median', int(_m.n_beats.median()), '| records with <3 beats:', int((_m.n_beats < 3).sum()),
      '| with 0 beats:', int((_m.n_beats == 0).sum()))
_leads = [LEADS.index(x) for x in ('II', 'V1', 'V2', 'V5')]
fig, ax = plt.subplots(3, 4, figsize=(9, 6.5))
for r, name in enumerate(CLASS_NAMES):
    ix = int(_m.index[_m.label == r][0])
    for c, l in enumerate(_leads):
        ax[r, c].imshow(_im[ix, 0, l], cmap='gray', vmin=0, vmax=255); ax[r, c].set_xticks([]); ax[r, c].set_yticks([])
        ax[r, c].set_title(f'{name} · {LEADS[l]}', fontsize=9)
fig.suptitle('V4 channel 0: signed median-beat traces (up = positive)'); fig.tight_layout(); plt.show()

## 3. Create train/validation/test dataloaders (no data leakage)

The dataset is read lazily from a memory-mapped uint8 `.npy` file; PyTorch receives one sample at a time in float32 range `[-1, 1]`. **Augmentation (random time shift, spectrogram time mask) is applied to the training loader only.** This avoids loading the multi-gigabyte cache fully into RAM. Train folds 1–8; validation fold 9. The test loader is created **only** during final evaluation. Printed class counts should match your previous V1/V2 cohort (8,377 / 1,052 / 1,073 records if using the same PTB-XL release and inclusion rules).

In [ ]:
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, roc_auc_score, precision_recall_fscore_support,
                             roc_curve, auc)

images = np.load(CACHE_DIR / 'images.npy', mmap_mode='r')
meta = pd.read_csv(CACHE_DIR / 'meta.csv')
assert tuple(images.shape) == (len(meta), *EXPECTED_IMAGE_SHAPE)
assert meta['codes'].isin(CLASS_NAMES).all()
assert meta['ecg_id'].is_unique
labels = meta['label'].to_numpy(dtype=np.int64)
folds = meta['strat_fold'].to_numpy(dtype=np.int64)
record_ids = meta['ecg_id'].to_numpy(dtype=np.int64)
assert set(labels) == {0, 1, 2}
train_idx = np.flatnonzero((folds >= 1) & (folds <= 8))
val_idx = np.flatnonzero(folds == 9)
test_idx = np.flatnonzero(folds == 10)
assert len(set(train_idx) & set(val_idx)) == 0
assert len(set(train_idx) & set(test_idx)) == 0
assert len(set(val_idx) & set(test_idx)) == 0
assert min(len(train_idx),len(val_idx),len(test_idx)) > 0
for section, idx in [('train',train_idx),('val',val_idx),('test (counts only)',test_idx)]:
    print(section, len(idx), dict(zip(CLASS_NAMES, np.bincount(labels[idx],minlength=3).tolist())))

# Hash the cohort's ordering, folds and labels so a resumed run cannot silently change data.
cohort_signature = hashlib.sha256(np.stack([record_ids, labels, folds], axis=1).astype('<i8').tobytes()).hexdigest()

class V4PortraitDataset(Dataset):
    def __init__(self, indices, train=False):
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        ix = int(self.indices[i])
        x = torch.from_numpy(np.array(images[ix], copy=True)).float().div_(127.5).sub_(1.0)   # (2,12,128,128)
        if self.train:      # torch RNG: seeded per DataLoader worker and restored on resume
            if AUG_SHIFT > 0:
                s0, s1 = (int(torch.randint(-AUG_SHIFT, AUG_SHIFT + 1, (1,))) for _ in range(2))
                x[0] = torch.roll(x[0], s0, dims=-1)
                x[1] = torch.roll(x[1], s1, dims=-1)
            if AUG_TIMEMASK > 0:
                w = int(torch.randint(0, AUG_TIMEMASK + 1, (1,)))
                if w:
                    st = int(torch.randint(0, x.shape[-1] - w + 1, (1,)))
                    x[1, :, :, st:st + w] = -1.0
        return x[0], x[1], int(labels[ix]), ix

def make_loader(indices, shuffle=False, train=False):
    return DataLoader(V4PortraitDataset(indices, train=train), batch_size=BATCH_SIZE,
                      shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=False)

val_loader = make_loader(val_idx)
# train_loader is created per seed in setup_run(); test_loader only during final evaluation

## 4. CPU training: weighted loss, EMA weights, gradient clipping, per-seed checkpoints and safe resumption

`setup_run(seed)` builds a fresh model, optimiser, scheduler and shuffled training loader for one seed and points `RUN_DIR` at `runs/mi_sspnet_v5_s<seed>/`. `train_or_resume()` then trains that seed.

- **EMA:** after every optimiser step, a shadow copy of the weights is updated as `ema = d·ema + (1−d)·weights` (with a short warm-up of `d`). Validation, checkpoint selection and `best.pt` all use the **EMA weights**. This removes much of the epoch-to-epoch noise seen in V3.
- **Checkpoint selection** on validation macro-AUC (`MONITOR`); the LR scheduler follows validation loss.
- `best.pt` (EMA weights), `best_val_predictions.npz` (fold-9 probabilities of that checkpoint, used by the ensemble), `last.pt` (full resumable state incl. EMA), `history.csv`, `config.json`.

In [ ]:
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

assert MONITOR in ('val_loss', 'val_macro_f1', 'val_macro_auc')
assert TOKEN_POOL in (1, 2, 4)
assert CLASS_WEIGHTING in ('none', 'sqrt', 'inverse')
assert MODEL_KIND in ('sspnet', 'cnn')
counts = np.bincount(labels[train_idx], minlength=3).astype(float)
if np.any(counts <= 0):
    raise ValueError('All 3 classes must have training recordings')
if CLASS_WEIGHTING == 'inverse':
    weights = counts.sum() / (3.0 * counts)
elif CLASS_WEIGHTING == 'sqrt':
    weights = np.sqrt(counts.sum() / (3.0 * counts))
else:
    weights = np.ones(3)
print('Training class weights:', dict(zip(CLASS_NAMES, weights.round(4))))
loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32), label_smoothing=LABEL_SMOOTHING)


def build_model():
    return (SSPNet(num_classes=3, dropout=DROPOUT, token_pool=TOKEN_POOL) if MODEL_KIND == 'sspnet'
            else BaselineCNN(num_classes=3, dropout=DROPOUT)).to(DEVICE)


def make_config(seed):
    return dict(version='sspnet-v5', token_pool=TOKEN_POOL, aug_shift=AUG_SHIFT, aug_timemask=AUG_TIMEMASK,
                use_ema=USE_EMA, ema_decay=EMA_DECAY, grad_clip=GRAD_CLIP,
                cache_schema=CACHE_SCHEMA, model=MODEL_KIND, seed=seed,
                input_shape=list(EXPECTED_IMAGE_SHAPE), cache=str(CACHE_DIR.resolve()),
                cohort_sha256=cohort_signature,
                batch_size=BATCH_SIZE, lr=LR, weight_decay=WEIGHT_DECAY,
                dropout=DROPOUT, label_smoothing=LABEL_SMOOTHING, weighting=CLASS_WEIGHTING,
                monitor=MONITOR, patience=PATIENCE, scheduler_patience=SCHEDULER_PATIENCE,
                scheduler_factor=SCHEDULER_FACTOR, classes=CLASS_NAMES)


class EMA:
    """Exponential moving average of all float tensors in the state dict (weights and BatchNorm statistics)."""
    def __init__(self, net, decay):
        self.decay, self.step = decay, 0
        self.model = copy.deepcopy(net).eval()
        for p in self.model.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, net):
        self.step += 1
        d = min(self.decay, (1 + self.step) / (10 + self.step))      # warm-up: early averages forget the random init
        src = net.state_dict()
        for k, v in self.model.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(d).add_(src[k].detach(), alpha=1 - d)
            else:
                v.copy_(src[k])


def setup_run(seed):
    """Fresh model/optimiser/scheduler/loader for one seed; sets the globals used by the training functions."""
    global SEED, RUN_DIR, config, model, ema, optimizer, scheduler, train_loader
    SEED, RUN_DIR = seed, run_dir_for(seed)
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    set_all_seeds(seed)
    config = make_config(seed)
    model = build_model()
    ema = EMA(model, EMA_DECAY) if USE_EMA else None
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=SCHEDULER_FACTOR, patience=SCHEDULER_PATIENCE)
    train_loader = make_loader(train_idx, shuffle=True, train=True)
    print(f'--- seed {seed} -> {RUN_DIR}')


def eval_model():
    return ema.model if ema is not None else model


def atomic_save(obj, path):
    path = Path(path)
    temp = path.with_name(path.name + '.tmp')
    torch.save(obj, temp)
    os.replace(temp, path)

def _metrics_from_prob(y_true, prob):
    pred = prob.argmax(axis=1)
    return dict(accuracy=float(accuracy_score(y_true, pred)),
                macro_f1=float(f1_score(y_true, pred, average='macro', zero_division=0)),
                macro_auc=float(roc_auc_score(y_true, prob, multi_class='ovr', average='macro')))

@torch.inference_mode()
def predict_loader(net, dl, calculate_loss=True):
    net.eval()
    y_chunks, p_chunks, id_chunks = [], [], []
    total_loss = 0.0
    for low, high, y, ix in dl:
        logits = net(low.to(DEVICE), high.to(DEVICE))
        if calculate_loss:
            total_loss += float(loss_fn(logits, y.to(DEVICE))) * len(y)
        y_chunks.append(y.numpy())
        p_chunks.append(torch.softmax(logits, dim=1).numpy())
        id_chunks.append(record_ids[ix.numpy()])
    y_true = np.concatenate(y_chunks)
    prob = np.concatenate(p_chunks)
    ids = np.concatenate(id_chunks)
    result = _metrics_from_prob(y_true, prob)
    result['loss'] = float(total_loss / len(y_true)) if calculate_loss else float('nan')
    return result, y_true, prob, ids

def train_one_epoch():
    model.train()
    total_loss, num = 0.0, 0
    for low, high, y, _ in train_loader:
        optimizer.zero_grad(set_to_none=True)
        logits = model(low.to(DEVICE), high.to(DEVICE))
        loss = loss_fn(logits, y.to(DEVICE))
        loss.backward()
        if GRAD_CLIP > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        if ema is not None:
            ema.update(model)
        total_loss += float(loss.detach()) * len(y)
        num += len(y)
    return total_loss / num

def training_snapshot(epoch, best_metric, best_epoch, bad, history, stopped):
    return dict(epoch=epoch, model=model.state_dict(), optimizer=optimizer.state_dict(),
                scheduler=scheduler.state_dict(), best_metric=best_metric,
                best_epoch=best_epoch, bad_epochs=bad, history=history,
                early_stopped=stopped, config=config,
                ema=(ema.model.state_dict() if ema is not None else None),
                ema_step=(ema.step if ema is not None else 0),
                torch_rng=torch.get_rng_state(), numpy_rng=np.random.get_state(),
                python_rng=random.getstate())

def train_or_resume():
    latest = RUN_DIR / 'last.pt'
    best = RUN_DIR / 'best.pt'
    epoch_start = 1
    best_metric = float('inf') if MONITOR == 'val_loss' else -float('inf')
    best_epoch, bad, history = 0, 0, []
    cf = RUN_DIR / 'config.json'
    if cf.exists():
        prev = json.loads(cf.read_text())
        if prev != config:
            raise ValueError(f'Run config in {RUN_DIR} differs. Change RUN_PREFIX for changed data/hyperparameters.')
    else:
        if latest.exists() or best.exists():
            raise FileExistsError('Checkpoint exists without config.json; inspect this directory first.')
        cf.write_text(json.dumps(config, indent=2))
    if latest.exists():
        if not RESUME:
            raise FileExistsError('Existing checkpoint. Set RESUME=True or change RUN_PREFIX.')
        cp = torch.load(latest, map_location='cpu', weights_only=False)   # only your own checkpoints
        if cp.get('config') != config:
            raise ValueError('Saved checkpoint is from another experiment.')
        model.load_state_dict(cp['model'])
        optimizer.load_state_dict(cp['optimizer'])
        scheduler.load_state_dict(cp['scheduler'])
        if ema is not None:
            ema.model.load_state_dict(cp['ema']); ema.step = int(cp['ema_step'])
        torch.set_rng_state(cp['torch_rng'])
        np.random.set_state(cp['numpy_rng'])
        random.setstate(cp['python_rng'])
        epoch_start = int(cp['epoch']) + 1
        best_metric, best_epoch = cp['best_metric'], cp['best_epoch']
        bad, history = cp['bad_epochs'], cp['history']
        print('Resumed after epoch', cp['epoch'], 'best epoch', best_epoch)
        if cp.get('early_stopped'):
            print('This run already early-stopped.')
            return history
    elif best.exists():
        raise RuntimeError('best.pt exists without last.pt; do not silently restart the run.')
    if epoch_start > EPOCHS:
        print('Run already reached configured EPOCHS.')
        return history
    for ep in range(epoch_start, EPOCHS + 1):
        t0 = time.time()
        train_loss = train_one_epoch()
        val, y_v, p_v, ids_v = predict_loader(eval_model(), val_loader)
        scheduler.step(val['loss'])
        score = val['loss'] if MONITOR == 'val_loss' else val[MONITOR[4:]]
        better = (score < best_metric - 1e-6) if MONITOR == 'val_loss' else (score > best_metric + 1e-6)
        if better:
            best_metric, best_epoch, bad = score, ep, 0
            atomic_save(eval_model().state_dict(), best)
            np.savez_compressed(RUN_DIR / 'best_val_predictions.npz',
                                y_true=y_v, prob=p_v, names=np.asarray(CLASS_NAMES), ecg_ids=ids_v)
        else:
            bad += 1
        entry = dict(epoch=ep, train_loss=train_loss, val_loss=val['loss'],
                     val_acc=val['accuracy'], val_macro_f1=val['macro_f1'],
                     val_macro_auc=val['macro_auc'], lr=optimizer.param_groups[0]['lr'],
                     elapsed_s=round(time.time() - t0, 1))
        history.append(entry)
        pd.DataFrame(history).to_csv(RUN_DIR / 'history.csv', index=False)
        stopped = bad >= PATIENCE
        atomic_save(training_snapshot(ep, best_metric, best_epoch, bad, history, stopped), latest)
        print(f'[s{SEED}] ep {ep:02d} train {train_loss:.4f} val {val["loss"]:.4f} '
              f'val_acc {val["accuracy"]:.4f} val_f1 {val["macro_f1"]:.4f} '
              f'val_auc {val["macro_auc"]:.4f} lr {optimizer.param_groups[0]["lr"]:.2e} '
              f'best_ep {best_epoch} bad {bad}/{PATIENCE} sec {entry["elapsed_s"]}', flush=True)
        if stopped:
            print('Early stopping at epoch', ep)
            break
    return history

## 5. Train or resume every seed (CPU)

Trains the seeds in `SEEDS` one after another. If you stop the kernel, rerunning this cell skips finished seeds and resumes the unfinished one from its last completed epoch. Expect several hours in total on CPU (watch the `sec` column); running it overnight is fine.

In [ ]:
if RUN_TRAINING:
    for s in SEEDS:
        setup_run(s)
        train_or_resume()
summary = []
for s in SEEDS:
    hf = run_dir_for(s) / 'history.csv'
    if hf.exists():
        h = pd.read_csv(hf); b = h.loc[h.val_macro_auc.idxmax()]
        summary.append(dict(seed=s, epochs=len(h), best_epoch=int(b.epoch), val_macro_auc=b.val_macro_auc,
                            val_macro_f1=b.val_macro_f1, val_acc=b.val_acc))
print(pd.DataFrame(summary).round(4).to_string(index=False) if summary else 'No seed has a history yet.')

## 6. Validation curves and class-wise diagnostics per seed (fold 9 ONLY)

Shows each finished seed separately. Fold 10 is not accessed. Figures go to each seed's `validation_figures/`.

In [ ]:
for SEED_SHOWN in SEEDS:
    RUN_DIR = run_dir_for(SEED_SHOWN)
    print('==== seed', SEED_SHOWN, RUN_DIR)
    import matplotlib.pyplot as plt
    if (RUN_DIR/'history.csv').exists():
        h=pd.read_csv(RUN_DIR/'history.csv')
        fig,axes=plt.subplots(1,3,figsize=(14,3.5))
        axes[0].plot(h.epoch,h.train_loss,label='Train loss')
        axes[0].plot(h.epoch,h.val_loss,label='Validation loss')
        axes[0].set(ylabel='Cross-entropy',xlabel='Epoch',title='Loss');axes[0].legend()
        axes[1].plot(h.epoch,h.val_acc*100,label='Val accuracy')
        axes[1].set(xlabel='Epoch',ylabel='Accuracy (%)',title='Validation accuracy')
        axes[2].plot(h.epoch,h.val_macro_f1,label='Val macro F1')
        axes[2].plot(h.epoch,h.val_macro_auc,label='Val macro AUC')
        axes[2].set(xlabel='Epoch',ylabel='Score',title='Validation macro F1 / AUC');axes[2].legend()
        fig.tight_layout();fig.savefig(RUN_DIR/'training_validation_overview.png',dpi=180);plt.show()
    else:
        print('No history.csv yet — train a model first.')

    if (RUN_DIR/'best_val_predictions.npz').is_file():
        d=np.load(RUN_DIR/'best_val_predictions.npz',allow_pickle=False)
        y_v,p_v=d['y_true'],d['prob']
        yhat=p_v.argmax(1)
        print('BEST checkpoint / validation fold 9 ONLY')
        print(classification_report(y_v,yhat,labels=np.arange(3),target_names=CLASS_NAMES,zero_division=0))
        vdir=RUN_DIR/'validation_figures';vdir.mkdir(exist_ok=True)
        cm=confusion_matrix(y_v,yhat,labels=np.arange(3))
        fig,ax=plt.subplots(figsize=(5.5,4.5))
        from sklearn.metrics import ConfusionMatrixDisplay
        ConfusionMatrixDisplay(cm,display_labels=CLASS_NAMES).plot(ax=ax,cmap='Blues',colorbar=False)
        ax.set_title('Validation fold 9 — best checkpoint')
        fig.tight_layout();fig.savefig(vdir/'validation_confusion.png',dpi=180);plt.show()
        fig,ax=plt.subplots(figsize=(6,5))
        for k,name in enumerate(CLASS_NAMES):
            fpr,tpr,_=roc_curve((y_v==k).astype(int),p_v[:,k])
            ax.plot(fpr,tpr,label=f'{name} AUC={auc(fpr,tpr):.3f}')
        ax.plot([0,1],[0,1],'--',color='gray')
        ax.set(xlabel='False positive rate',ylabel='True positive rate',title='Validation fold 9 — one-vs-rest ROC')
        ax.legend();fig.tight_layout();fig.savefig(vdir/'validation_roc.png',dpi=180);plt.show()
    else:
        print('No best_val_predictions.npz yet — train a model first.')

## 6b. Seed ensemble and validation-tuned class offsets (fold 9 ONLY)

**Ensemble:** the validation probabilities of all finished seeds (from their `best_val_predictions.npz`) are averaged. Nothing is retrained.

**Class offsets:** argmax on imbalanced data favours NORM. The decision rule becomes `argmax_k [ log p_k + b_k ]` with `b_NORM = 0` and `b_IMI, b_ASMI` chosen on a coarse grid to maximise **validation macro-F1**. Only 2 numbers are learned from fold 9, and they are then frozen for fold 10. This trades some NORM specificity for MI sensitivity, which is the right direction for an MI detector. With only 34 ASMI validation records the offsets are somewhat noisy, so the grid is deliberately coarse (step 0.1), and ties are broken towards smaller offsets.

AUC is ranking-based and almost unaffected by the offsets; accuracy, F1, sensitivity and the confusion matrix change.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

def apply_bias(prob, bias):
    z = np.log(np.clip(prob, 1e-12, 1.0)) + np.asarray(bias)[None, :]
    z -= z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)

def tune_bias(y, prob, grid=BIAS_GRID):
    """Grid search over (b_IMI, b_ASMI), b_NORM = 0, maximising macro-F1; ties -> smallest |b|."""
    logp = np.log(np.clip(prob, 1e-12, 1.0))
    best = (-1.0, 0.0, (0.0, 0.0, 0.0))
    for b1 in grid:
        for b2 in grid:
            pred = (logp + np.array([0.0, b1, b2])).argmax(1)
            f = f1_score(y, pred, average='macro', zero_division=0)
            key = (round(f, 6), -(abs(b1) + abs(b2)))
            if key > (round(best[0], 6), best[1]):
                best = (f, -(abs(b1) + abs(b2)), (0.0, float(b1) + 0.0, float(b2) + 0.0))
    return np.array(best[2]), best[0]

def summarize(y, prob, tag):
    pred = prob.argmax(1)
    rec = (confusion_matrix(y, pred, labels=np.arange(3)).diagonal()
                                    / np.bincount(y, minlength=3))
    return dict(model=tag, macro_auc=roc_auc_score(y, prob, multi_class='ovr', average='macro'),
                macro_f1=f1_score(y, pred, average='macro', zero_division=0),
                accuracy=accuracy_score(y, pred),
                **{f'recall_{n}': r for n, r in zip(CLASS_NAMES, rec)})

members = [s for s in SEEDS if (run_dir_for(s) / 'best_val_predictions.npz').is_file()
           and (run_dir_for(s) / 'best.pt').is_file()]
if not members:
    print('No finished seed yet - train first.')
else:
    rows, probs, y_ref, ids_ref = [], [], None, None
    for s in members:
        d = np.load(run_dir_for(s) / 'best_val_predictions.npz', allow_pickle=False)
        if y_ref is None:
            y_ref, ids_ref = d['y_true'], d['ecg_ids']
        elif not (np.array_equal(ids_ref, d['ecg_ids']) and np.array_equal(y_ref, d['y_true'])):
            raise ValueError(f'Seed {s}: validation records/order differ from the first member.')
        probs.append(d['prob'])
        rows.append(summarize(y_ref, d['prob'], f'seed {s}'))
    p_ens = np.mean(probs, axis=0)
    rows.append(summarize(y_ref, p_ens, f'ensemble ({len(members)} seeds)'))
    bias, f1_tuned = tune_bias(y_ref, p_ens)
    p_ens_b = apply_bias(p_ens, bias)
    rows.append(summarize(y_ref, p_ens_b, 'ensemble + offsets'))
    tab = pd.DataFrame(rows)
    print('VALIDATION fold 9 (offsets tuned here, so the last row is optimistic):')
    print(tab.round(4).to_string(index=False))
    print('offsets (NORM, IMI, ASMI):', bias)
    if len(members) > 1:
        seed_rows = tab.iloc[:len(members)]
        print(f"single-seed macro-AUC mean +- SD: {seed_rows.macro_auc.mean():.4f} +- {seed_rows.macro_auc.std(ddof=1):.4f}")
    print(classification_report(y_ref, p_ens_b.argmax(1), labels=np.arange(3), target_names=CLASS_NAMES, zero_division=0))

    ENSEMBLE_DIR.mkdir(parents=True, exist_ok=True)
    (ENSEMBLE_DIR / 'ensemble_config.json').write_text(json.dumps(
        dict(members=members, run_dirs=[str(run_dir_for(s)) for s in members], bias=bias.tolist(),
             val_macro_f1_tuned=f1_tuned, bias_grid=[float(BIAS_GRID.min()), float(BIAS_GRID.max()), 0.1],
             classes=CLASS_NAMES), indent=2))
    tab.round(4).to_csv(ENSEMBLE_DIR / 'validation_summary.csv', index=False)
    np.savez_compressed(ENSEMBLE_DIR / 'ensemble_val_predictions.npz', y_true=y_ref, prob=p_ens_b,
                        prob_raw=p_ens, names=np.asarray(CLASS_NAMES), ecg_ids=ids_ref)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
    for ax, p, t in [(axes[0], p_ens, 'Ensemble, argmax'), (axes[1], p_ens_b, 'Ensemble + offsets')]:
        ConfusionMatrixDisplay(confusion_matrix(y_ref, p.argmax(1), labels=np.arange(3)),
                               display_labels=CLASS_NAMES).plot(ax=ax, cmap='Blues', colorbar=False)
        ax.set_title(f'Validation fold 9 - {t}')
    fig.tight_layout(); fig.savefig(ENSEMBLE_DIR / 'validation_confusion_ensemble.png', dpi=180); plt.show()
    print('Ensemble definition frozen in', ENSEMBLE_DIR / 'ensemble_config.json',
          '(rerun this cell if more seeds finish BEFORE the test evaluation).')

## 7. FINAL fold-10 evaluation (disabled by default)

**Run once, after all decisions are frozen** (seeds, hyperparameters, ensemble members, offsets in `ensemble_config.json`). Set `EVALUATE_TEST = True` and run this cell.

It evaluates every ensemble member on fold 10 (writing `predictions.npz` / `results.json` in each seed directory, used for the mean ± SD across seeds), then averages them, applies the frozen offsets, and writes `predictions.npz` (`prob` = with offsets, `prob_raw` = without), `results.json` and `results_raw.json` to `runs/mi_sspnet_v5_ensemble/`.

In [ ]:
if not EVALUATE_TEST:
    print('Protected test fold 10: NOT evaluated. Set EVALUATE_TEST=True only for the final locked experiment.')
else:
    ecfg = json.loads((ENSEMBLE_DIR / 'ensemble_config.json').read_text())   # run Section 6b first
    if (ENSEMBLE_DIR / 'predictions.npz').exists():
        raise FileExistsError('Final ensemble results already exist. Avoid repeated testing.')
    test_loader = make_loader(test_idx)
    probs, y_ref, ids_ref = [], None, None
    for s in ecfg['members']:
        rd = run_dir_for(s)
        net = build_model()
        net.load_state_dict(torch.load(rd / 'best.pt', map_location='cpu', weights_only=True))
        m, y_true, prob, ids = predict_loader(net, test_loader, calculate_loss=False)
        if y_ref is None:
            y_ref, ids_ref = y_true, ids
        assert np.array_equal(ids, ids_ref)
        probs.append(prob)
        scores = {k: m[k] for k in ['accuracy', 'macro_f1', 'macro_auc']}
        print(f'seed {s} TEST:', {k: round(v, 4) for k, v in scores.items()})
        if not (rd / 'predictions.npz').exists():
            np.savez_compressed(rd / 'predictions.npz', y_true=y_true, prob=prob,
                                names=np.asarray(CLASS_NAMES), ecg_ids=ids)
            (rd / 'results.json').write_text(json.dumps(scores, indent=2))
    p_raw = np.mean(probs, axis=0)
    p_adj = apply_bias(p_raw, ecfg['bias'])
    raw_scores = _metrics_from_prob(y_ref, p_raw)
    adj_scores = _metrics_from_prob(y_ref, p_adj)
    print('\nENSEMBLE TEST (argmax):', json.dumps(raw_scores, indent=2))
    print('ENSEMBLE TEST (+ frozen offsets):', json.dumps(adj_scores, indent=2))
    print(classification_report(y_ref, p_adj.argmax(1), labels=np.arange(3), target_names=CLASS_NAMES, zero_division=0))
    print('Confusion matrix (rows true):')
    print(confusion_matrix(y_ref, p_adj.argmax(1), labels=np.arange(3)))
    np.savez_compressed(ENSEMBLE_DIR / 'predictions.npz', y_true=y_ref, prob=p_adj, prob_raw=p_raw,
                        names=np.asarray(CLASS_NAMES), ecg_ids=ids_ref)
    (ENSEMBLE_DIR / 'results.json').write_text(json.dumps(adj_scores, indent=2))
    (ENSEMBLE_DIR / 'results_raw.json').write_text(json.dumps(raw_scores, indent=2))
    # Figure 1 needs a training history: use the member with the best validation AUC, and say so in the caption.
    best_member = max(ecfg['members'], key=lambda s: pd.read_csv(run_dir_for(s) / 'history.csv').val_macro_auc.max())
    for f in ['history.csv', 'config.json']:
        shutil.copy(run_dir_for(best_member) / f, ENSEMBLE_DIR / f)
    print('Saved to', ENSEMBLE_DIR, '| Fig. 1 training curves are from seed', best_member)

## 8. Full publication-style figures and metric tables — **original functions retained**

Includes the original **Fig. 1** training/validation curves, **Fig. 2** normalized confusion matrix, **Fig. 3** class-wise ROC curves, **Fig. 4** precision/recall/F1, **Fig. 5** optional literature comparison, tables with 95% bootstrap CIs (`metrics_table.csv`, `.tex`), and PNG + vector PDF exports. The literature figure compares different models/protocols and must **not** be interpreted as a controlled benchmark.

The exact figure functions from the original notebook are included in the next cell. Figure 1 marks the selected best epoch according to the run's configured checkpoint criterion (V4 default: validation macro-AUC).

In [ ]:
"""
Publication-style figures + metric tables from a finished train_ptbxl.py run.

Reads  <run>/history.csv  and  <run>/predictions.npz  (written by train_ptbxl.py)
Writes <run>/figures/
    fig1_training_curves   loss (train/val) and validation accuracy per epoch
    fig2_confusion_matrix  counts + row-normalised fractions on the test fold
    fig3_roc_curves        one-vs-rest ROC per class with AUC
    fig4_per_class_metrics precision / recall / F1 per class
    fig5_literature_*      per-class AUC vs Chen et al. 2021 (only if IMI/ASMI/... are among your classes)
    metrics_table.csv/.tex accuracy, macro-F1, macro-AUC and per-class AUC with 95% bootstrap CIs
    per_class_table.csv    precision / recall / F1 / support
    figure_captions.md     draft captions
Every figure is saved as PNG (300 dpi) and vector PDF.

Usage:
  python make_figures.py --run runs/mi
"""
import argparse
import json
from pathlib import Path

import matplotlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support,
                             roc_auc_score, roc_curve)

# ---- style: white surface, ink text, recessive grid, colour-blind-validated categorical order ----
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e4e3df"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]   # blue, orange, aqua, yellow, magenta
LINESTYLES = ["-", "--", "-.", ":", (0, (5, 1))]                    # secondary encoding besides colour
MARKERS = ["o", "s", "^", "D", "v"]
SEQ = LinearSegmentedColormap.from_list("seq", ["#ffffff", "#2a78d6"])

# Per-class AUCs reported by Chen et al. 2021, Front. Cardiovasc. Med. 8:654515 (PTB-XL holdout, 1D ResNet)
LIT_AUC = {"IMI": 0.973, "ASMI": 0.987, "AMI": 0.961, "LMI": 0.969, "ALMI": 0.996}

plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 300, "savefig.bbox": "tight", "pdf.fonttype": 42,
    "font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"], "font.size": 10,
    "text.color": INK, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
    "axes.edgecolor": INK2, "axes.linewidth": 0.8, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "legend.frameon": False, "legend.fontsize": 9, "lines.linewidth": 2.0,
})


def save(fig, base):
    fig.savefig(f"{base}.png")
    fig.savefig(f"{base}.pdf")
    plt.close(fig)
    print("wrote", f"{base}.png / .pdf")


def macro_auc(y, prob, K):
    return roc_auc_score(y, prob[:, 1]) if K == 2 else roc_auc_score(y, prob, multi_class="ovr", average="macro")


def bootstrap(y, prob, K, n, seed=0):
    rng, N, rows = np.random.default_rng(seed), len(y), []
    for _ in range(n):
        i = rng.integers(0, N, N)
        yb, pb = y[i], prob[i]
        if len(np.unique(yb)) < K:
            continue
        pred = pb.argmax(1)
        r = {"accuracy": accuracy_score(yb, pred), "macro_f1": f1_score(yb, pred, average="macro"),
             "macro_auc": macro_auc(yb, pb, K)}
        for k in range(K):
            r[f"auc_{k}"] = roc_auc_score(yb == k, pb[:, k])
        rows.append(r)
    return pd.DataFrame(rows)


# ------------------------------------------------------------------ figures
def fig_curves(hist, out, monitor="val_loss"):
    fig, (a, b) = plt.subplots(1, 2, figsize=(7.2, 3.2))
    a.plot(hist.epoch, hist.train_loss, color=SERIES[0], label="Training")
    a.plot(hist.epoch, hist.val_loss, color=SERIES[1], linestyle="--", label="Validation")
    # Preserve the original validation-loss rule; also support optional macro-F1 runs.
    if monitor in ('val_macro_f1', 'val_macro_auc') and monitor in hist.columns:
        best = int(hist.epoch[hist[monitor].idxmax()])
    else:
        best = int(hist.epoch[hist.val_loss.idxmin()])
    for ax in (a, b):
        ax.axvline(best, color=INK2, lw=0.8, ls=":")
        ax.set_xlabel("Epoch")
    a.set_ylabel("Cross-entropy loss")
    a.legend(loc="upper right")
    a.text(best, 0.5, f"best epoch {best} ", transform=a.get_xaxis_transform(), ha="right", va="center",
           rotation=90, color=INK2, fontsize=8)
    b.plot(hist.epoch, hist.val_acc * 100, color=SERIES[0])
    b.set_ylabel("Validation accuracy (%)")
    a.set_title("(a) Loss", loc="left", fontsize=10)
    b.set_title("(b) Validation accuracy", loc="left", fontsize=10)
    fig.tight_layout()
    save(fig, out / "fig1_training_curves")


def fig_confusion(cm, names, out):
    K = len(names)
    frac = cm / cm.sum(1, keepdims=True)
    fig, ax = plt.subplots(figsize=(1.6 + 0.9 * K, 1.4 + 0.9 * K))
    im = ax.imshow(frac, cmap=SEQ, vmin=0, vmax=1)
    ax.grid(False)
    ax.set_xticks(range(K), names)
    ax.set_yticks(range(K), names)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    for i in range(K):
        for j in range(K):
            ax.text(j, i, f"{cm[i, j]}\n({frac[i, j] * 100:.1f}%)", ha="center", va="center", fontsize=9,
                    color="white" if frac[i, j] > 0.55 else INK)
    for s in ax.spines.values():
        s.set_visible(False)
    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cb.set_label("Fraction of true class")
    cb.outline.set_visible(False)
    save(fig, out / "fig2_confusion_matrix")


def fig_roc(y, prob, names, out):
    K = len(names)
    fig, ax = plt.subplots(figsize=(4.4, 4.2))
    ax.plot([0, 1], [0, 1], color=INK2, lw=0.9, ls=":", label="Chance")
    for k in range(K):
        fpr, tpr, _ = roc_curve(y == k, prob[:, k])
        auc = roc_auc_score(y == k, prob[:, k])
        ax.plot(fpr, tpr, color=SERIES[k % 5], ls=LINESTYLES[k % 5], label=f"{names[k]} (AUC {auc:.3f})")
    ax.set_xlabel("False positive rate (1 - specificity)")
    ax.set_ylabel("True positive rate (sensitivity)")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.01)
    ax.set_aspect("equal")
    ax.legend(loc="lower right")
    save(fig, out / "fig3_roc_curves")


def fig_per_class(df, out):
    K = len(df)
    fig, ax = plt.subplots(figsize=(1.8 + 1.4 * K, 3.4))
    w = 0.26
    x = np.arange(K)
    for m, (col, lab) in enumerate([("precision", "Precision"), ("recall", "Recall (sensitivity)"), ("f1", "F1-score")]):
        ax.bar(x + (m - 1) * w, df[col], w, color=SERIES[m], edgecolor="white", linewidth=1.2, label=lab)
    ax.set_xticks(x, df["class"])
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score")
    ax.grid(axis="x", visible=False)
    ax.legend(loc="lower center", ncol=3, bbox_to_anchor=(0.5, 1.0))
    save(fig, out / "fig4_per_class_metrics")


def fig_literature(auc_ours, names, out):
    common = [n for n in names if n in LIT_AUC]
    if not common:
        return False
    x, w = np.arange(len(common)), 0.34
    fig, ax = plt.subplots(figsize=(1.8 + 1.5 * len(common), 3.4))
    ax.bar(x - w / 2, [auc_ours[names.index(n)] for n in common], w, color=SERIES[0], edgecolor="white",
           linewidth=1.2, label="This work (SSPNet, spectrogram bands)")
    ax.bar(x + w / 2, [LIT_AUC[n] for n in common], w, color=SERIES[1], edgecolor="white", linewidth=1.2,
           label="Chen et al. 2021 (1D ResNet)")
    ax.set_xticks(x, common)
    ax.set_ylim(0.5, 1.0)
    ax.set_ylabel("One-vs-rest AUC")
    ax.grid(axis="x", visible=False)
    ax.legend(loc="lower center", ncol=1, bbox_to_anchor=(0.5, 1.0))
    save(fig, out / "fig5_literature_comparison")
    return True


# ------------------------------------------------------------------ main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--run", required=True, help="run directory written by train_ptbxl.py")
    ap.add_argument("--boot", type=int, default=1000, help="bootstrap resamples for 95%% CIs")
    args = ap.parse_args()

    run = Path(args.run)
    out = run / "figures"
    out.mkdir(exist_ok=True)
    hist = pd.read_csv(run / "history.csv")
    d = np.load(run / "predictions.npz", allow_pickle=False)
    y, prob, names = d["y_true"], d["prob"], [str(n) for n in d["names"]]
    K, pred = len(names), d["prob"].argmax(1)

    cm = confusion_matrix(y, pred, labels=range(K))
    p, r, f, s = precision_recall_fscore_support(y, pred, labels=range(K), zero_division=0)
    per = pd.DataFrame({"class": names, "precision": p, "recall": r, "f1": f, "support": s,
                        "auc": [roc_auc_score(y == k, prob[:, k]) for k in range(K)]})
    per.round(4).to_csv(out / "per_class_table.csv", index=False)

    cf = run / 'config.json'
    try:
        selected_monitor = json.loads(cf.read_text()).get('monitor', 'val_loss') if cf.exists() else 'val_loss'
    except (ValueError, OSError):
        selected_monitor = 'val_loss'
    fig_curves(hist, out, monitor=selected_monitor)
    fig_confusion(cm, names, out)
    fig_roc(y, prob, names, out)
    fig_per_class(per, out)
    has_lit = fig_literature(per["auc"].tolist(), names, out)

    # ---- metric table with 95% bootstrap CIs
    bs = bootstrap(y, prob, K, args.boot)
    point = {"accuracy": accuracy_score(y, pred), "macro_f1": f1_score(y, pred, average="macro"),
             "macro_auc": macro_auc(y, prob, K)}
    point.update({f"auc_{k}": per["auc"][k] for k in range(K)})
    labels = {"accuracy": "Accuracy", "macro_f1": "Macro F1", "macro_auc": "Macro AUC (OvR)"}
    labels.update({f"auc_{k}": f"AUC {names[k]}" for k in range(K)})
    rows = [dict(metric=labels[m], value=point[m], ci_low=bs[m].quantile(0.025), ci_high=bs[m].quantile(0.975))
            for m in labels]
    tab = pd.DataFrame(rows)
    tab.round(4).to_csv(out / "metrics_table.csv", index=False)
    with open(out / "metrics_table.tex", "w") as fh:
        fh.write("\\begin{tabular}{lc}\n\\hline\nMetric & Value [95\\% CI] \\\\\n\\hline\n")
        for _, t in tab.iterrows():
            fh.write(f"{t.metric} & {t.value:.3f} [{t.ci_low:.3f}, {t.ci_high:.3f}] \\\\\n")
        fh.write("\\hline\n\\end{tabular}\n")
    print(tab.round(3).to_string(index=False))

    cap = [
        f"**Fig. 1.** Training dynamics of SSPNet on PTB-XL ({', '.join(names)}). (a) Training and validation "
        "cross-entropy loss; the dotted line marks the checkpoint selected by validation "
        + {"val_macro_f1": "macro F1", "val_macro_auc": "macro AUC"}.get(selected_monitor, "loss")
        + ", whose weights were used for testing. (b) Validation accuracy.",
        f"**Fig. 2.** Confusion matrix on the held-out PTB-XL test fold (strat_fold 10, n = {len(y)}). Cells show "
        "counts and the percentage of each true class.",
        "**Fig. 3.** One-vs-rest receiver operating characteristic curves on the test fold; the dotted line is "
        "chance.",
        "**Fig. 4.** Per-class precision, recall and F1-score on the test fold.",
        f"**Table 1.** Test-fold performance with 95% bootstrap confidence intervals ({args.boot} resamples of "
        "ECG records).",
    ]
    if has_lit:
        cap.append("**Fig. 5.** One-vs-rest AUC per myocardial-infarction location compared with Chen et al. (2021). "
                   "Protocols differ (their model is a 1D ResNet trained for MI localisation on PTB-XL), so this is "
                   "an indicative comparison, not a controlled one.")
    (out / "figure_captions.md").write_text("\n\n".join(cap) + "\n")
    print("done ->", out)


main_figures = main      # entry point, called from the run cell below

In [ ]:
# Make original figures for a completed model (new V3 or any previous run).
# Defaults to the V5 ensemble; use run_dir_for(1) for a single seed or V3_BASELINE_RUN for V3.
FIGURE_RUN=ENSEMBLE_DIR
BOOTSTRAP_RESAMPLES=1000
from IPython.display import Image, display
if not (FIGURE_RUN/'predictions.npz').exists():
    print('No final test predictions in',FIGURE_RUN,'; test figures intentionally skipped.')
else:
    saved_argv=sys.argv[:]
    try:
        sys.argv=['make_figures','--run',str(FIGURE_RUN),'--boot',str(BOOTSTRAP_RESAMPLES)]
        main_figures()
    finally:
        sys.argv=saved_argv
    for name in ['fig1_training_curves','fig2_confusion_matrix','fig3_roc_curves',
                 'fig4_per_class_metrics','fig5_literature_comparison']:
        f=FIGURE_RUN/'figures'/(name+'.png')
        if f.exists(): display(Image(filename=str(f),width=680))

## 9. Full statistics — **original DeLong, McNemar, Holm, bootstrap support retained**

The original statistical functions are included below: per-class AUC vs chance (DeLong, Holm-adjusted), exact one-sided binomial comparison with majority-class accuracy, paired **McNemar exact** test and paired **DeLong difference** versus baseline, and multi-seed mean ± SD. The **bootstrap 95% confidence intervals** are generated by Section 8.

**Pairing safeguard:** V3 and V4 store test `ecg_ids` in `predictions.npz`. Paired V3-vs-V3 comparisons require **exact matching IDs/order**. Old V1/V2 predictions may not store IDs; comparison with those must be explicitly opted into *only after verifying that both test record orders truly match*. Identical label arrays alone are not proof that ECG records correspond.

In [ ]:
"""
Significance tests for a finished train_ptbxl.py run (uses <run>/predictions.npz).

Single model
  * per-class AUC vs 0.5 (chance)            DeLong z-test
  * accuracy vs majority-class baseline       exact one-sided binomial test
Compare with another model (--baseline runs/other; same test fold, same classes)
  * per-class AUC difference                  DeLong test for two correlated AUCs
  * accuracy difference                       McNemar exact test on paired predictions
Several seeds (--seed-runs runs/mi_s1 runs/mi_s2 ...)
  * mean +- SD of accuracy / macro-F1 / macro-AUC from each run's results.json
P-values for the per-class tests are Holm-adjusted across classes.

Usage:
  python stats_tests.py --run runs/mi
  python stats_tests.py --run runs/mi --baseline runs/resnet_baseline
  python stats_tests.py --seed-runs runs/mi_s1 runs/mi_s2 runs/mi_s3
"""
import argparse
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import binomtest, norm


# ---------------------------------------------------------------- DeLong (Sun & Xu 2014 fast algorithm)
def _midrank(x):
    order = np.argsort(x)
    z = x[order]
    n, ranks, i = len(x), np.zeros(len(x)), 0
    while i < n:
        j = i
        while j < n and z[j] == z[i]:
            j += 1
        ranks[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    out = np.empty(n)
    out[order] = ranks
    return out


def delong(y_bin, scores):
    """y_bin: (n,) 0/1; scores: (k, n) scores of k models. Returns AUCs (k,) and their covariance (k, k)."""
    scores = np.atleast_2d(scores)
    order = np.argsort(-y_bin, kind="stable")           # positives first
    s = scores[:, order]
    m = int(y_bin.sum())
    n = s.shape[1] - m
    k = s.shape[0]
    tx, ty, tz = np.empty((k, m)), np.empty((k, n)), np.empty((k, m + n))
    for r in range(k):
        tx[r], ty[r], tz[r] = _midrank(s[r, :m]), _midrank(s[r, m:]), _midrank(s[r])
    aucs = tz[:, :m].sum(1) / m / n - (m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    cov = np.atleast_2d(np.cov(v01)) / m + np.atleast_2d(np.cov(v10)) / n
    return aucs, cov


def delong_vs_chance(y_bin, score):
    auc, cov = delong(y_bin, score)
    z = (auc[0] - 0.5) / np.sqrt(cov[0, 0])
    return auc[0], 2 * norm.sf(abs(z))


def delong_two(y_bin, s1, s2):
    aucs, cov = delong(y_bin, np.vstack([s1, s2]))
    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    z = (aucs[0] - aucs[1]) / np.sqrt(var) if var > 0 else 0.0
    return aucs[0], aucs[1], 2 * norm.sf(abs(z))


def mcnemar_exact(correct_a, correct_b):
    b = int(np.sum(correct_a & ~correct_b))     # A right, B wrong
    c = int(np.sum(~correct_a & correct_b))     # A wrong, B right
    p = 1.0 if b + c == 0 else binomtest(b, b + c, 0.5).pvalue
    return b, c, p


def holm(p):
    p = np.asarray(p, float)
    order = np.argsort(p)
    adj = np.empty_like(p)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (len(p) - rank) * p[i])
        adj[i] = min(1.0, running)
    return adj


def fmt_p(p):
    return "<0.001" if p < 0.001 else f"{p:.3f}"


def load(run):
    d = np.load(Path(run) / "predictions.npz", allow_pickle=False)
    return d["y_true"], d["prob"], [str(n) for n in d["names"]]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--run", help="run directory (needs predictions.npz)")
    ap.add_argument("--baseline", help="second run directory to compare against (same test fold)")
    ap.add_argument("--seed-runs", nargs="+", help="run directories of repeated seeds to aggregate")
    args = ap.parse_args()

    if args.seed_runs:
        rows = [json.load(open(Path(r) / "results.json")) for r in args.seed_runs]
        df = pd.DataFrame(rows)[["accuracy", "macro_f1", "macro_auc"]]
        print(f"\n{len(rows)} seeds (mean +- SD):")
        for c in df:
            print(f"  {c:10s} {df[c].mean():.4f} +- {df[c].std(ddof=1) if len(df) > 1 else 0:.4f}")
    if not args.run:
        return

    run = Path(args.run)
    y, prob, names = load(run)
    K, pred = len(names), prob.argmax(1)

    # ---- single-model tests
    rows = []
    for k in range(K):
        auc, p = delong_vs_chance((y == k).astype(int), prob[:, k])
        rows.append(dict(class_=names[k], auc=auc, p_vs_chance=p))
    tab = pd.DataFrame(rows)
    tab["p_holm"] = holm(tab["p_vs_chance"])
    acc = float(np.mean(pred == y))
    maj = np.bincount(y, minlength=K).max() / len(y)
    p_acc = binomtest(int((pred == y).sum()), len(y), maj, alternative="greater").pvalue
    print(f"\nAccuracy {acc:.4f} vs majority-class rate {maj:.4f}: one-sided exact binomial p {fmt_p(p_acc)}")
    print("\nPer-class AUC vs chance (DeLong, Holm-adjusted):")
    print(tab.assign(auc=tab.auc.round(4), p_vs_chance=tab.p_vs_chance.map(fmt_p),
                     p_holm=tab.p_holm.map(fmt_p)).to_string(index=False))
    report = [tab.assign(test="AUC vs 0.5")]

    # ---- comparison with another model
    if args.baseline:
        y2, prob2, names2 = load(args.baseline)
        if names2 != names or not np.array_equal(y, y2):
            raise SystemExit("baseline must use the same classes and the same test records/order")
        pred2 = prob2.argmax(1)
        b, c, p_mc = mcnemar_exact(pred == y, pred2 == y)
        print(f"\nvs baseline: accuracy {acc:.4f} vs {np.mean(pred2 == y):.4f}; "
              f"McNemar exact (A-only right {b}, B-only right {c}) p {fmt_p(p_mc)}")
        rows = []
        for k in range(K):
            a1, a2, p = delong_two((y == k).astype(int), prob[:, k], prob2[:, k])
            rows.append(dict(class_=names[k], auc_this=a1, auc_baseline=a2, p_diff=p))
        cmp_ = pd.DataFrame(rows)
        cmp_["p_holm"] = holm(cmp_["p_diff"])
        print("\nPer-class AUC difference (DeLong, Holm-adjusted):")
        print(cmp_.assign(auc_this=cmp_.auc_this.round(4), auc_baseline=cmp_.auc_baseline.round(4),
                          p_diff=cmp_.p_diff.map(fmt_p), p_holm=cmp_.p_holm.map(fmt_p)).to_string(index=False))
        report.append(cmp_.assign(test="AUC vs baseline", mcnemar_p=p_mc))

    out = run / "stats_report.csv"
    pd.concat(report, ignore_index=True).to_csv(out, index=False)
    print("\nsaved", out)


main_stats = main      # entry point, called from the run cell below

In [ ]:
# Statistics on completed runs, using the original routines.
STAT_RUN=ENSEMBLE_DIR
# Default: paired comparison of V4 against your V3 SSPNet run (same test records). For a plain-CNN comparison use
# Path('runs/mi_cnn_v4_s1') after training MODEL_KIND='cnn' with its own RUN_DIR.
BASELINE_RUN=V3_BASELINE_RUN
ALLOW_LEGACY_ORDER_MATCH=False   # keep False unless you have independently verified old ECG ordering

def _paired_ids_ok(run_a,run_b):
    a=np.load(Path(run_a)/'predictions.npz',allow_pickle=False)
    b=np.load(Path(run_b)/'predictions.npz',allow_pickle=False)
    if 'ecg_ids' in a.files and 'ecg_ids' in b.files:
        if not np.array_equal(a['ecg_ids'],b['ecg_ids']):
            raise ValueError('Paired test rejected: ECG record IDs/order differ.')
        return True
    if ALLOW_LEGACY_ORDER_MATCH:
        print('WARNING: no ecg_ids available in at least one run; relying on user-verified order.')
        return True
    print('Paired comparison skipped: ecg_ids missing in one/both runs. '
          'Use matching V3 baseline, or explicitly verify legacy IDs/order.')
    return False

if not (STAT_RUN/'predictions.npz').exists():
    print('No test predictions; statistics skipped for',STAT_RUN)
else:
    cli=['stats_tests','--run',str(STAT_RUN)]
    if (BASELINE_RUN/'predictions.npz').exists() and _paired_ids_ok(STAT_RUN,BASELINE_RUN):
        cli += ['--baseline',str(BASELINE_RUN)]
    prev=sys.argv[:]
    try:
        sys.argv=cli
        main_stats()
    finally:
        sys.argv=prev

In [ ]:
# Mean +- SD across the individual seeds (each seed's fold-10 results.json, written by Section 7).
finished = [run_dir_for(s) for s in SEEDS if (run_dir_for(s) / 'results.json').is_file()]
if len(finished) >= 2:
    old_argv = sys.argv[:]
    try:
        sys.argv = ['stats_tests', '--seed-runs', *[str(r) for r in finished]]
        main_stats()
    finally:
        sys.argv = old_argv
    print('Ensemble (with offsets):', (ENSEMBLE_DIR / 'results.json').read_text() if (ENSEMBLE_DIR / 'results.json').exists() else 'n/a')
else:
    print('Multi-seed summary skipped: fewer than two seeds with final test results.')

## 10. Reporting notes, experiment controls and limitations

- **What to report:** (i) mean ± SD over seeds for single models, (ii) the seed ensemble, (iii) the ensemble with validation-tuned offsets. Never report only the best seed: seed choice by test (or even validation) score is selection bias.
- **What V5 changes vs V4:** EMA weights, gradient clipping, multi-seed ensemble, decision offsets tuned on fold 9. Inputs, cohort, folds, loss, optimiser and architecture are as in V4.
- **Offsets:** two numbers learned on fold 9 (34 ASMI / 104 IMI records), then frozen. They shift the operating point towards MI sensitivity; AUC is essentially unchanged by them. Report both argmax and offset results.
- **Test-fold disclosure:** V4/V5 were designed after V1–V3 results were inspected, including fold 10. State this; report fold 10 once for the final V5 configuration; external validation would strengthen the claim.
- **Preprocessing limits:** the median beat assumes a mostly regular rhythm; fixed ±1.5 mV scale clips very large QRS; 10 s at 100 Hz.
- **Research caveat:** not a clinical diagnostic device.